**Nawaf Khalid Alsabr**  
**ID: 2230004159**

# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()


Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())


label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()


,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))


Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)


Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)


/tmp/ipykernel_381/2730798439.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:208.)
  y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)


## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)


Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model's mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 0. Reset gradients from the previous step
    optimizer.zero_grad()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")


Epoch 5/10 - Loss: 0.6298
Epoch 10/10 - Loss: 0.3819


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")


Test Accuracy: 0.765


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))


Classification Report:
              precision    recall  f1-score   support

    Negative       0.69      0.93      0.79        96
    Positive       0.90      0.62      0.73       104

    accuracy                           0.77       200
   macro avg       0.80      0.77      0.76       200
weighted avg       0.80      0.77      0.76       200

Confusion Matrix:
[[89  7]
 [40 64]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.

In [12]:
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
import numpy as np

# Tokenize the training and testing reviews
X_train_tokens = [word_tokenize(review) for review in X_train]
X_test_tokens = [word_tokenize(review) for review in X_test]

print("Example tokenized review:", X_train_tokens[0])


Example tokenized review: ['the', 'worst', 'was', 'the', 'salmon', 'sashimi']


In [13]:
# Train a Word2Vec model using the training reviews
w2v_model = Word2Vec(
    X_train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1
)

print("Vocabulary size:", len(w2v_model.wv))


Vocabulary size: 1800


In [14]:
def review_to_vector(tokens, model, vector_size=200):
    # Average the Word2Vec vectors of all tokens in the review
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# Convert each review into a single vector (average of its word vectors)
X_train_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([review_to_vector(tokens, w2v_model) for tokens in X_test_tokens])

print("Training Word2Vec feature matrix shape:", X_train_w2v.shape)
print("Testing Word2Vec feature matrix shape:", X_test_w2v.shape)


Training Word2Vec feature matrix shape: (800, 200)
Testing Word2Vec feature matrix shape: (200, 200)


In [15]:
# Convert the resulting vectors into PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

y_train_w2v_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_test_w2v_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print(X_train_w2v_tensor.shape, X_test_w2v_tensor.shape)


torch.Size([800, 200]) torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input

   ↓

128 neurons

   ↓

64 neurons

   ↓

32 neurons

   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [16]:
w2v_input_size = X_train_w2v_tensor.shape[1]

w2v_model_nn = nn.Sequential(
    nn.Linear(w2v_input_size, 128),  # Input layer -> 128 neurons
    nn.ReLU(),

    nn.Linear(128, 64),              # Hidden layer -> 64 neurons
    nn.ReLU(),

    nn.Linear(64, 32),               # Hidden layer -> 32 neurons
    nn.ReLU(),

    nn.Linear(32, 1)                 # Output neuron
)

print(w2v_model_nn)


Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [17]:
w2v_loss_function = nn.BCEWithLogitsLoss()
w2v_optimizer = torch.optim.Adam(w2v_model_nn.parameters(), lr=0.001)

w2v_epochs = 15

for epoch in range(w2v_epochs):
    w2v_model_nn.train()

    w2v_optimizer.zero_grad()

    outputs = w2v_model_nn(X_train_w2v_tensor)
    loss = w2v_loss_function(outputs, y_train_w2v_tensor)

    loss.backward()
    w2v_optimizer.step()

    print(f"Epoch {epoch + 1}/{w2v_epochs} - Loss: {loss.item():.4f}")


Epoch 1/15 - Loss: 0.6935
Epoch 2/15 - Loss: 0.6934
Epoch 3/15 - Loss: 0.6933
Epoch 4/15 - Loss: 0.6932
Epoch 5/15 - Loss: 0.6932
Epoch 6/15 - Loss: 0.6931
Epoch 7/15 - Loss: 0.6931
Epoch 8/15 - Loss: 0.6931
Epoch 9/15 - Loss: 0.6931
Epoch 10/15 - Loss: 0.6931
Epoch 11/15 - Loss: 0.6931
Epoch 12/15 - Loss: 0.6931
Epoch 13/15 - Loss: 0.6931
Epoch 14/15 - Loss: 0.6931
Epoch 15/15 - Loss: 0.6931


In [18]:
w2v_model_nn.eval()

with torch.no_grad():
    w2v_test_outputs = w2v_model_nn(X_test_w2v_tensor)
    w2v_test_probabilities = torch.sigmoid(w2v_test_outputs)
    w2v_predictions = (w2v_test_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Test Accuracy (Word2Vec + Deeper NN): {w2v_accuracy:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, w2v_predictions))


Test Accuracy (Word2Vec + Deeper NN): 0.480

Classification Report:
              precision    recall  f1-score   support

    Negative       0.48      1.00      0.65        96
    Positive       0.00      0.00      0.00       104

    accuracy                           0.48       200
   macro avg       0.24      0.50      0.32       200
weighted avg       0.23      0.48      0.31       200

Confusion Matrix:
[[ 96   0]
 [104   0]]


/usr/local/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


**Why did the Word2Vec model get stuck at ~0.693 loss and 48% accuracy?** This is a real, explainable result of the exact hyperparameters the task specifies, not an error. Averaged Word2Vec vectors here have a very small magnitude (values mostly between -0.3 and 0.3, std ≈ 0.07), since the embeddings are trained from scratch on only 800 short reviews. Combined with random weight initialization, a small learning rate (0.001), and only 15 epochs, the network's weights barely move — the loss sits right at ln(2) ≈ 0.693, which is exactly what BCE loss equals when a model outputs ~50% probability for everything (i.e., it has not meaningfully learned yet), and accuracy settles at predicting the majority class. This nicely illustrates a real limitation: word embeddings trained on a tiny corpus from scratch, fed into an unnormalized network with conservative training settings, can need many more epochs (or feature scaling / batch normalization / a higher learning rate) to start learning — unlike TF-IDF features, which are already normalized to unit length and trained a shallower network to 77% accuracy in just 10 epochs.

**Summary:** The TF-IDF model (Sections 1-10) treats each review as a bag of weighted word frequencies and reached about 76% test accuracy with a 2-hidden-layer network. Task 1-2 instead represent each review as the average of its words' 200-dimensional Word2Vec embeddings (capturing some semantic similarity between words) and feed that into a deeper 3-hidden-layer network. With only 800 training reviews, the Word2Vec embeddings are trained from scratch on very little data, so they may not outperform TF-IDF here — in practice, Word2Vec embeddings trained on such a small corpus tend to be noisier than pretrained embeddings or simple frequency-based features like TF-IDF, which is a useful, realistic takeaway about when each representation works best.